# Prompt template

In [23]:
PROMPT_TEMPLATE = """
You are an expert in Causal Inference. Your task is to determine the causal direction between two variables based on their context and metadata.

Dataset Context: {context}
Variable X: {var_x_desc}
Variable Y: {var_y_desc}

Based on physical laws, common sense, and scientific facts, select the most plausible causal direction:
- X -> Y (X causes Y)
- Y -> X (Y causes X)
- Independent (No direct causal relationship)

Strict Output Format (DO NOT use any markdown, do not use double asterisks ** anywhere):
Direction: [Your choice: X -> Y, Y -> X, or Independent]
Reason: [Provide a brief explanation in 1-2 sentences]
"""

# Ten pairs of Tuebingen

In [24]:
TUEBINGEN_PAIRS = [
    {
        "pair_id": "0001",
        "context": "DWD data (Deutscher Wetterdienst) data was taken at 349 stations",
        "var_x":"altitude",
        "var_y":"temperature (average over 1961-1990)",
        "ground_truth": "X -> Y"

    },
    {
            "pair_id": "0002",
            "context": "DWD data (Deutscher Wetterdienst) data was taken at 349 stations",
            "var_x":"altitude",
            "var_y":"precipitation (yearly value averaged over 1961-1990)",
            "ground_truth": "X -> Y"
    },
    {
            "pair_id": "0003",
            "context": "DWD data (Deutscher Wetterdienst) data was taken at 349 stations",
            "var_x":"longitude",
            "var_y":"temperature (averaged over 1961-1990)",
            "ground_truth": "X -> Y"
    
    },
    {
            "pair_id": "0004",
            "context": "DWD data (Deutscher Wetterdienst) data was taken at 349 stations",
            "var_x":"altitude",
            "var_y":"sunshine (yearly value averaged over 1961-1990)",
            "ground_truth": "X -> Y"
    },
    {
            "pair_id": "0005",
            "context": "Abalone data",
            "var_x":"Rings",
            "var_y":"Length",
            "ground_truth": "X -> Y"
    },
    {
            "pair_id": "0006",
            "context": "Abalone data",
            "var_x":"Rings",
            "var_y":"Shell weight",
            "ground_truth": "X -> Y"
    },
    {
            "pair_id": "0007",
            "context": "Abalone data",
            "var_x":"Rings",
            "var_y":"Diameter",
            "ground_truth": "X -> Y"
    },
    {
            "pair_id": "0008",
            "context": "Abalone data",
            "var_x":"Rings",
            "var_y":"Height",
            "ground_truth": "X -> Y"
    },
    {
            "pair_id": "0009",
            "context": "Abalone data",
            "var_x":"Rings",
            "var_y":"Whole weight",
            "ground_truth": "X -> Y"
    },
    {
            "pair_id": "0010",
            "context": "Abalone data",
            "var_x":"Rings",
            "var_y":"Shucked weight",
            "ground_truth": "X -> Y"
    }
]

# Run LLM for causal predictions

In [25]:
import ollama

def run_llama(prompt, model="llama3.1:8b"):
    response = ollama.chat(
        model=model,
        messages=[
            {
                "role": "user",
                "content": prompt,
            }
        ],
        options={
            "temperature": 0,
            "top_p": 1,
            "top_k": 1,
            "seed": 42,
        },
        stream=False,
    )

    output = response["message"]["content"]

    predicted_direction = "N/A"
    reason = "N/A"

    for line in output.splitlines():
        if line.startswith("Direction:"):
            predicted_direction = line.replace("Direction:", "").strip()
        elif line.startswith("Reason:"):
            reason = line.replace("Reason:", "").strip()

    return {
        "output": output,
        "predicted_direction": predicted_direction,
        "reason": reason,
    }

In [26]:
import csv

csv_data = []
for pair in TUEBINGEN_PAIRS :
    prompt = PROMPT_TEMPLATE.format(
        context=pair["context"],
        var_x_desc=pair["var_x"],
        var_y_desc=pair["var_y"])

    result = run_llama(prompt)

    predicted_direction = result["predicted_direction"]
    reason = result["reason"]

    correctness = ( pair["ground_truth"] in predicted_direction
        or predicted_direction in pair["ground_truth"]
    )
    
    csv_data.append({
        "Pair_ID": pair["pair_id"],
        "Variable_X": pair["var_x"],
        "Variable_Y": pair["var_y"],
        "Predicted_Direction": predicted_direction,
        "Ground_Truth": pair["ground_truth"],
        "Correctness": correctness,
        "Reason": reason
    })

csv_file_name = "../output/causal_predictions.csv"
headers = ["Pair_ID", "Variable_X", "Variable_Y", "Ground_Truth", "Predicted_Direction", "Correctness", "Reason"]

with open(csv_file_name, mode="w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=headers)
    writer.writeheader()
    writer.writerows(csv_data)

print(f"Finish write to file: {csv_file_name}")

Finish write to file: ../output/causal_predictions.csv


# Neutral and Paraphrased, then compare in 1 table

In [32]:
PROMPT_TEMPLATE_PARAPHRASED = """
Your role is to perform causal reasoning. Using the provided context and metadata, decide which variable is the most likely cause of the other.

Dataset Context: {context}
Variable X: {var_x_desc}
Variable Y: {var_y_desc}

Based on the information provided, identify the causal direction between the two variables:
- X -> Y (X causes Y)
- Y -> X (Y causes X)
- Independent (No direct causal relationship)

Strict Output Format (DO NOT use any markdown, do not use double asterisks ** anywhere):
Direction: [Your choice: X -> Y, Y -> X, or Independent]
Reason: [Provide a brief explanation in 1-2 sentences]
"""

result_normal = []
for pair in TUEBINGEN_PAIRS :
    prompt = PROMPT_TEMPLATE.format(
        context=pair["context"],
        var_x_desc=pair["var_x"],
        var_y_desc=pair["var_y"])

    result = run_llama(prompt)

    predicted_direction = result["predicted_direction"]
    reason = result["reason"]

    correctness = ( pair["ground_truth"] in result["predicted_direction"]
        or result["predicted_direction"] in pair["ground_truth"]
    )
    
    result_normal.append({
        "pair_id": pair["pair_id"],
        "var_x": pair["var_x"],
        "var_y": pair["var_y"],
        "ground_truth": pair["ground_truth"],
        "predicted_direction": predicted_direction,
        "correctness": correctness,
        "reason": reason
    })

result_neutral = []
for pair in TUEBINGEN_PAIRS :
    prompt = PROMPT_TEMPLATE.format(
        context=pair["context"],
        var_x_desc= "A",
        var_y_desc= "B")

    result = run_llama(prompt)

    predicted_direction = result["predicted_direction"]
    reason = result["reason"]

    correctness = ( pair["ground_truth"] in result["predicted_direction"]
        or result["predicted_direction"] in pair["ground_truth"]
    )

    # add result
    result_neutral.append({
        "pair_id": pair["pair_id"],
        "predicted_direction":predicted_direction,
        "correctness": correctness,
        "reason": reason
    })

result_paraphrased = []
for pair in TUEBINGEN_PAIRS :
    prompt = PROMPT_TEMPLATE_PARAPHRASED.format(
        context=pair["context"],
        var_x_desc=pair["var_x"],
        var_y_desc=pair["var_y"])

    result = run_llama(prompt)

    predicted_direction = result["predicted_direction"]
    reason = result["reason"]

    correctness = ( pair["ground_truth"] in result["predicted_direction"]
        or result["predicted_direction"] in pair["ground_truth"]
    )
    # add result
    result_paraphrased.append({
        "pair_id": pair["pair_id"],
        "predicted_direction":predicted_direction,
        "correctness": correctness,
        "reason": reason
    })

comparision = []
result_normal_map = {
    item["pair_id"]: item for item in result_normal
}
result_neutral_map = {
    item["pair_id"]: item for item in result_neutral
}
result_paraphrased_map = {
    item["pair_id"]: item for item in result_paraphrased
}
for item in result_normal:
    pair_id = item["pair_id"]
    org_direction = result_normal_map[pair_id]["predicted_direction"]
    neutral_direction = result_neutral_map[pair_id]["predicted_direction"]
    paraphrased_direction = result_paraphrased_map[pair_id]["predicted_direction"]
    comparision.append({
        "Pair": item["pair_id"],
        "Var_X": item["var_x"],
        "Var_Y": item["var_y"],
        "Ground_Truth": item["ground_truth"],
        "Original": org_direction,
        "Neutral(A/B) ": neutral_direction,
        "Paraphrased": paraphrased_direction,
        "Flip?(Original ~ Neutral)": not ( org_direction in neutral_direction or neutral_direction in org_direction),
        "Flip?(Original ~ Paraphrased)": not ( org_direction in paraphrased_direction or paraphrased_direction in org_direction)
    })

import pandas as pd 

df = pd.DataFrame(comparision)

df

,Pair,Var_X,Var_Y,Ground_Truth,Original,Neutral(A/B),Paraphrased,Flip?(Original ~ Neutral),Flip?(Original ~ Paraphrased)
0,0001,altitude,temperature (average over 1961-1990),X -> Y,X -> Y,X -> Y,Y -> X,False,True
1,0002,altitude,precipitation (yearly value averaged over 1961...,X -> Y,X -> Y,X -> Y,X -> Y,False,False
2,0003,longitude,temperature (averaged over 1961-1990),X -> Y,Y -> X,X -> Y,Y -> X,True,False
3,0004,altitude,sunshine (yearly value averaged over 1961-1990),X -> Y,X -> Y,X -> Y,X -> Y,False,False
4,0005,Rings,Length,X -> Y,X -> Y,X -> Y,X -> Y,False,False
5,0006,Rings,Shell weight,X -> Y,X -> Y,X -> Y,X -> Y,False,False
6,0007,Rings,Diameter,X -> Y,X -> Y,X -> Y,Y -> X,False,True
7,0008,Rings,Height,X -> Y,X -> Y,X -> Y,X -> Y,False,False
8,0009,Rings,Whole weight,X -> Y,X -> Y,X -> Y,X -> Y,False,False
9,0010,Rings,Shucked weight,X -> Y,X -> Y,X -> Y,X -> Y,False,False
